In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q10: TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT
q10 = spark.sql('''
WITH state_changes AS (
    SELECT
        date,
        timestamp,
        COMP,
        LAG(COMP,1,COMP) OVER(ORDER BY timestamp) AS prev_comp
    FROM sensor
)
SELECT
    date,
    COUNT(*) AS so_lan_dong_ngat
FROM state_changes
WHERE COMP <> prev_comp
GROUP BY date
ORDER BY so_lan_dong_ngat DESC
LIMIT 10
''')
print("\n========== EXECUTION PLAN ==========")
q10.explain(True)
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT ---")
df_q10 = q10.toPandas()
try:
    display(df_q10)
except NameError:
    print(df_q10.to_string(index=False))